# Aula 05 — Gradient checking e paridade com NumPy

[Aula](../aulas/05-gradcheck-paridade-numpy.md) · [Currículo](../README.md)

## Objetivo e método

Auditar uma MLP fixa por três caminhos: backward manual NumPy, autograd PyTorch e diferenças centrais. As contraprovas devem detectar defeitos; não basta obter um teste positivo.

Dados sintéticos, sem ajuste ou avaliação preditiva. Não há necessidade de splits: medimos derivadas locais, não generalização. As entradas e os pesos são gerados uma vez em NumPy e copiados para PyTorch. Nenhuma API de modelo ou otimizador é necessária.

## Preparação

Mínimos: Python 3.10, NumPy 1.24 e PyTorch 2.6. Para abrir localmente, use Jupyter; nbformat 5.10 permite validar o formato. Instale dependências compatíveis com sua plataforma antes de executar. Não há downloads, GPU ou credenciais.

Execute em kernel novo, na ordem. CPU, float64 e seed `20260905`. As únicas entradas float32 são as contraprovas de precisão explicitamente identificadas.

In [ ]:
import platform
import numpy as np
import torch

SEED = 20260905
rng = np.random.default_rng(SEED)
checks = []

def check(name, condition):
    assert bool(condition), name
    checks.append(name)

def close(name, actual, expected, atol=1e-12, rtol=1e-12):
    actual, expected = np.asarray(actual), np.asarray(expected)
    assert actual.shape == expected.shape, name + ": shape"
    assert np.isfinite(actual).all() and np.isfinite(expected).all(), name + ": finitude"
    np.testing.assert_allclose(actual, expected, atol=atol, rtol=rtol)
    checks.append(name)

print("Python", platform.python_version(), "NumPy", np.__version__, "PyTorch", torch.__version__)
check("precisão dupla", np.dtype('float64').itemsize == 8)

## 1. Fixture com 26 parâmetros

$B=5$, $D=3$, $H=4$, $C=2$. Rede afim → tanh → afim. Loss: metade da média sobre todos os $BC=10$ resíduos quadráticos. Pesos pequenos evitam uma fixture dominada pela saturação.

In [ ]:
B, D, H, C = 5, 3, 4, 2
X = rng.normal(size=(B, D))
T = rng.normal(size=(B, C))
names = ('W1', 'b1', 'W2', 'b2')
params = [rng.normal(scale=0.2, size=(D, H)), rng.normal(scale=0.1, size=H),
          rng.normal(scale=0.2, size=(H, C)), rng.normal(scale=0.1, size=C)]
baseline = [p.copy() for p in params]

def numpy_forward(inputs, targets, ps):
    w1, b1, w2, b2 = ps
    z = inputs @ w1 + b1
    a = np.tanh(z)
    s = a @ w2 + b2
    assert s.shape == targets.shape
    loss = float(np.mean((s - targets) ** 2) / 2)
    return loss, (z, a, s)

def numpy_backward(inputs, targets, ps):
    loss, (z, a, s) = numpy_forward(inputs, targets, ps)
    ds = (s - targets) / targets.size
    dz = (ds @ ps[2].T) * (1 - a ** 2)
    grads = [inputs.T @ dz, dz.sum(axis=0), a.T @ ds, ds.sum(axis=0)]
    return loss, grads

loss_np, grads_np = numpy_backward(X, T, params)
check("26 coordenadas", sum(p.size for p in params) == 26)
check("shapes manuais", all(p.shape == g.shape for p, g in zip(params, grads_np)))
print("Loss NumPy:", loss_np)

## 2. Mesmos bytes de entrada, outra implementação

`torch.tensor` copia os arrays, cria folhas e usa float64 explicitamente. A loss será uma função dos quatro parâmetros; X e T são constantes da fixture.

In [ ]:
Xt = torch.tensor(X, dtype=torch.float64)
Tt = torch.tensor(T, dtype=torch.float64)

def torch_params():
    return tuple(torch.tensor(p, dtype=torch.float64, requires_grad=True) for p in params)

def torch_forward(*ps):
    w1, b1, w2, b2 = ps
    z = Xt @ w1 + b1
    a = torch.tanh(z)
    s = a @ w2 + b2
    assert s.shape == Tt.shape
    return (s - Tt).square().mean() / 2, (z, a, s)

def objective(*ps):
    return torch_forward(*ps)[0]

pt = torch_params()
loss_t, cache_t = torch_forward(*pt)
gt = torch.autograd.grad(loss_t, pt)
grads_t = [g.detach().numpy().copy() for g in gt]
close("loss PyTorch/NumPy", loss_t.item(), loss_np)
for name, nt, nn in zip(('Z', 'A', 'S'), cache_t, numpy_forward(X, T, params)[1]):
    close("forward " + name, nt.detach().numpy(), nn)
for name, actual, expected in zip(names, grads_t, grads_np):
    close("backward " + name, actual, expected)
check("grad retorna sem acumular", all(p.grad is None for p in pt))
parity_error = max(np.max(np.abs(a - b)) for a, b in zip(grads_t, grads_np))
print("Maior erro manual/autograd:", parity_error)

## 3. Diferenças centrais por coordenada

Cada consulta recebe cópias independentes dos parâmetros. Para cada coordenada, o passo é $h_j=h\max(1,|\theta_j|)$. A função retorna 26 estimativas, sem alterar os parâmetros de referência.

In [ ]:
def central_all(h):
    estimates = []
    for block, parameter in enumerate(params):
        estimate = np.empty_like(parameter)
        for index in np.ndindex(parameter.shape):
            step = h * max(1.0, abs(parameter[index]))
            plus, minus = [p.copy() for p in params], [p.copy() for p in params]
            plus[block][index] += step
            minus[block][index] -= step
            fp = numpy_forward(X, T, plus)[0]
            fm = numpy_forward(X, T, minus)[0]
            estimate[index] = (fp - fm) / (2 * step)
        estimates.append(estimate)
    return estimates

numeric = central_all(1e-5)
numeric_error = max(np.max(np.abs(a - b)) for a, b in zip(numeric, grads_np))
for name, actual, expected in zip(names, numeric, grads_np):
    close("diferença central " + name, actual, expected, atol=1e-8, rtol=1e-6)
check("fixture não foi mutada", all(np.array_equal(a, b) for a, b in zip(params, baseline)))
print("Erro máximo das diferenças centrais:", numeric_error)

## 4. Relatório por bloco e tolerância mista

O maior erro absoluto localiza discrepâncias; a razão de normas resume a escala do bloco. O critério elemento a elemento usado aqui é $|a-b|\leq10^{-8}+10^{-6}|b|$, com b como referência manual. Shapes e finitude são conferidos antes.

In [ ]:
report = []
for name, actual, expected in zip(names, numeric, grads_np):
    error = np.abs(actual - expected)
    relative = np.linalg.norm(actual - expected) / max(
        1e-12, np.linalg.norm(actual) + np.linalg.norm(expected))
    worst = np.unravel_index(np.argmax(error), error.shape)
    passed = bool(np.all(error <= 1e-8 + 1e-6 * np.abs(expected)))
    report.append((name, float(error.max()), float(relative), worst, passed))
    check("critério misto " + name, passed)
for row in report:
    print(row)
# Um erro absoluto pequeno pode ser enorme em escala relativa perto de zero.
near_a, near_b = 1e-12, -1e-12
check("tolerância absoluta perto de zero", abs(near_a - near_b) <= 1e-8)
check("razão relativa isolada enganosa", abs(near_a - near_b) / (abs(near_a) + abs(near_b)) == 1)

## 5. Varredura de h e cancelamento

Não escolhemos uma tolerância nova para cada resultado. Mantemos a referência e inspecionamos o efeito de h. A segunda parte mostra uma perturbação de $10^{-8}$ que deixa de ser representável ao redor de 1 em float32.

In [ ]:
sweep = []
for h in (1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11):
    candidates = central_all(h)
    error = max(np.max(np.abs(a - b)) for a, b in zip(candidates, grads_np))
    sweep.append((h, float(error)))
    print(f"h={h:.0e}, erro={error:.6e}")
check("região intermediária melhor que passo grande", sweep[2][1] < sweep[0][1])
check("passo extremo pior que intermediário", sweep[-1][1] > sweep[2][1])
one32 = np.float32(1)
h32 = np.float32(1e-8)
estimate32 = float(((one32 + h32) ** 3 - (one32 - h32) ** 3) / (2 * h32))
estimate64 = ((1.0 + 1e-8) ** 3 - (1.0 - 1e-8) ** 3) / 2e-8
check("perturbações float32 arredondadas", one32 + h32 == one32 and one32 - h32 == one32)
close("diferença float32 zero", estimate32, 0.0)
close("diferença float64 próxima de 3", estimate64, 3.0, atol=1e-7)
print("Derivada do cubo: float32", estimate32, "float64", estimate64)

## 6. `torch.autograd.gradcheck`

Passamos todos os parâmetros diferenciáveis na tupla. A closure fixa somente os dados. Usamos `fast_mode=False` para a auditoria pequena completa, `eps=1e-6`, `atol=1e-5`, `rtol=1e-3` e `nondet_tol=0.0`, valores declarados da API 2.6.

In [ ]:
pt_check = torch_params()
before = [p.detach().clone() for p in pt_check]
gradcheck_ok = torch.autograd.gradcheck(
    objective, pt_check, eps=1e-6, atol=1e-5, rtol=1e-3,
    nondet_tol=0.0, fast_mode=False, raise_exception=True)
check("gradcheck MLP", gradcheck_ok)
check("gradcheck preserva parâmetros", all(torch.equal(p.detach(), q)
      for p, q in zip(pt_check, before)))
print("gradcheck MLP:", gradcheck_ok)

## 7. Cinco direções como verificação adicional

Cada direção conjunta tem norma 1 sobre os 26 parâmetros. Comparamos a diferença central direcional e a soma dos produtos internos dos gradientes manuais. Isso é um complemento: uma direção pode ser ortogonal a um defeito.

In [ ]:
direction_errors = []
for trial in range(5):
    directions = [rng.normal(size=p.shape) for p in params]
    norm = np.sqrt(sum(np.sum(d*d) for d in directions))
    directions = [d / norm for d in directions]
    h = 1e-5
    fp = numpy_forward(X, T, [p+h*d for p, d in zip(params, directions)])[0]
    fm = numpy_forward(X, T, [p-h*d for p, d in zip(params, directions)])[0]
    numerical = (fp-fm)/(2*h)
    analytical = sum(np.sum(g*d) for g, d in zip(grads_np, directions))
    close(f"direção {trial}", numerical, analytical, atol=1e-8, rtol=1e-6)
    direction_errors.append(abs(numerical-analytical))
print("Maior erro direcional:", max(direction_errors))

## 8. Localizar um defeito no backward manual

Omitimos deliberadamente o fator $(1-A^2)$ antes da primeira camada. O forward continua idêntico e os shapes também. Os gradientes de W2/b2 devem continuar corretos, enquanto W1/b1 falham.

In [ ]:
_, (_, activation, scores) = numpy_forward(X, T, params)
ds = (scores - T) / T.size
bad_dz = ds @ params[2].T
bad_grads = [X.T @ bad_dz, bad_dz.sum(axis=0), activation.T @ ds, ds.sum(axis=0)]
bad_errors = [float(np.max(np.abs(a-b))) for a, b in zip(bad_grads, grads_t)]
check("defeito W1 detectado", bad_errors[0] > 1e-4)
check("defeito b1 detectado", bad_errors[1] > 1e-4)
close("W2 correto apesar do defeito", bad_grads[2], grads_t[2])
close("b2 correto apesar do defeito", bad_grads[3], grads_t[3])
print(dict(zip(names, bad_errors)))

## 9. Gradcheck pode aprovar o objetivo errado

Multiplicar nossa loss por C equivale a dividir a soma somente por B. Autograd diferencia essa outra função corretamente; gradcheck deve passar. A referência NumPy do objetivo contratado é quem detecta o fator 2.

In [ ]:
wrong_objective = lambda *ps: objective(*ps) * C
wrong_inputs = torch_params()
check("objetivo errado internamente consistente",
      torch.autograd.gradcheck(wrong_objective, wrong_inputs))
wrong_derivatives = torch.autograd.grad(wrong_objective(*wrong_inputs), wrong_inputs)
for name, actual, reference in zip(names, wrong_derivatives, grads_np):
    close("fator C " + name, actual.detach().numpy(), C * reference)
check("objetivo errado difere do contrato", not np.isclose(C * loss_np, loss_np))
print("Loss contratada:", loss_np, "loss com redução errada:", C * loss_np)

## 10. Contraprova com backward fornecido errado

Esta pequena `Function` é apenas uma injeção de defeito: o forward é $x^2$, mas o backward devolve $3x$ vezes a sensibilidade recebida. `ctx` guarda a entrada para a consulta e `apply` executa a operação. Não é necessário criar Functions para a MLP usual.

In [ ]:
class WrongSquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.square()

    @staticmethod
    def backward(ctx, upstream):
        x, = ctx.saved_tensors
        return upstream * 3 * x  # defeito intencional: deveria ser 2*x

sample = torch.tensor([0.4, -0.7], dtype=torch.float64, requires_grad=True)
check("quadrado nativo aprovado", torch.autograd.gradcheck(torch.square, (sample,)))
check("backward defeituoso rejeitado", not torch.autograd.gradcheck(
    WrongSquare.apply, (sample,), raise_exception=False))
print("Quadrado nativo: True; backward defeituoso: False")

## 11. Quina e memória sobreposta

Em zero, ReLU não é diferenciável: a diferença central vale 1/2, mas PyTorch usa derivada 0. Longe da quina, a comparação passa. Depois, verificamos que `expand` cria coordenadas com armazenamento compartilhado; fazer clone permite tratá-las como entradas independentes.

In [ ]:
origin = torch.tensor([0.0], dtype=torch.float64, requires_grad=True)
check("ReLU na quina não passa", not torch.autograd.gradcheck(
    torch.relu, (origin,), raise_exception=False))
away = torch.tensor([-0.7, 0.4], dtype=torch.float64, requires_grad=True)
check("ReLU fora da quina passa", torch.autograd.gradcheck(torch.relu, (away,)))
relu_autograd, = torch.autograd.grad(torch.relu(origin).sum(), origin)
close("convenção na quina", relu_autograd.numpy(), np.array([0.0]))
check("diferença central na quina", (max(0, 1e-6)-max(0, -1e-6))/2e-6 == 0.5)
expanded = torch.tensor([0.5], dtype=torch.float64, requires_grad=True).expand(3)
check("stride zero denuncia sobreposição", expanded.stride() == (0,))
independent = expanded.detach().clone().requires_grad_()
check("clone com coordenadas independentes", independent.stride() == (1,))
check("clone passa gradcheck", torch.autograd.gradcheck(torch.square, (independent,)))
print("ReLU: derivada autograd 0; diferença central 0.5; longe da quina: True")

## 12. Aleatoriedade congelada e atualização equivalente

Uma máscara Bernoulli fixa define uma função determinística durante toda a auditoria. Reamostrar a máscara mudaria o objeto comparado. Finalmente, um único passo explícito de SGD verifica também o sinal e a escala da atualização, sem treinar nem estimar desempenho.

In [ ]:
mask = torch.tensor(rng.binomial(1, 0.7, size=(5,)), dtype=torch.float64) / 0.7
drop_input = torch.tensor(rng.normal(size=5), dtype=torch.float64, requires_grad=True)
frozen_dropout = lambda x: (x * mask).square().sum()
check("máscara congelada reprodutível", torch.equal(frozen_dropout(drop_input), frozen_dropout(drop_input)))
check("máscara congelada passa", torch.autograd.gradcheck(frozen_dropout, (drop_input,)))
eta = 0.05
numpy_next = [p - eta*g for p, g in zip(params, grads_np)]
torch_next = [p.detach() - eta*g for p, g in zip(pt, gt)]
for name, actual, expected in zip(names, torch_next, numpy_next):
    close("atualização " + name, actual.numpy(), expected)
step_error = max(float(np.max(np.abs(a.numpy()-b))) for a, b in zip(torch_next, numpy_next))
print("Maior erro após uma atualização:", step_error)

## Verificações finais

Todos os resultados são locais à fixture e às tolerâncias declaradas. O teste por parâmetro e o gradcheck são completos para estes quatro blocos; X e T foram tratados como constantes. Nenhuma conclusão é feita sobre derivadas complexas, funções arbitrárias, GPU ou generalização.

In [ ]:
check("contratos com nomes únicos", len(checks) == len(set(checks)))
print(f"{len(checks)} contratos aprovados")
print({"loss": loss_np, "parity_error": float(parity_error),
       "numeric_error": float(numeric_error), "direction_error": float(max(direction_errors)),
       "step_error": step_error, "coordinates": sum(p.size for p in params)})

## Exercícios e próximos passos

1. Troque mean por sum. **Resposta:** se o objetivo original era a média sobre 10 resíduos, loss e gradientes aumentam dez vezes; gradcheck ainda pode aprovar a função alterada.
2. Teste ReLU em 0.4 com passo $10^{-6}$. **Resposta:** as perturbações continuam no ramo positivo e as duas derivadas são aproximadamente 1.
3. Injete uma média indevida no gradiente do bias. **Resposta:** o relatório por bloco identifica o bias, mesmo com shapes corretos.
4. Congele uma máscara diferente e repita a auditoria. **Resposta:** os gradientes podem mudar, mas a consistência local ainda deve passar.

Próxima aula: `nn.Module`, parâmetros e buffers. Antes de organizar o código em módulos, já temos uma referência numérica verificável.

### Fontes técnicas

Verificadas em 9 de setembro de 2026: [Gradcheck mechanics — PyTorch 2.6](https://docs.pytorch.org/docs/2.6/notes/gradcheck.html), [gradcheck — API 2.6](https://docs.pytorch.org/docs/2.6/generated/torch.autograd.gradcheck.gradcheck.html), [Numerical accuracy — PyTorch 2.6](https://docs.pytorch.org/docs/2.6/notes/numerical_accuracy.html) e [assert_allclose — NumPy 2.3](https://numpy.org/doc/2.3/reference/generated/numpy.testing.assert_allclose.html).